In [29]:
import os
import pycolmap
import numpy as np
from tqdm.notebook import tqdm
from mylib.geometry import compute_relative_camera_motion

# pairs_calibrated.txt is used for pose estimation benchmarks
# views.txt is used only for repeatability benchmarks

In [30]:
data_path = '/home/mattia/Desktop/datasets/mydataset/data_test'
scenes = sorted(os.listdir(data_path))
# scenes if scene is a directory]  # filter only valid
scenes = [scene for scene in scenes if os.path.isdir(os.path.join(data_path, scene))]
# remove graz_castle, graz_clocktower, graz_main_square
scenes = [scene for scene in scenes if scene not in ['graz_castle', 'graz_clocktower', 'graz_main_square']]

len(scenes), scenes 

(12,
 ['graz_church',
  'graz_townhall',
  'graz_university',
  'munich_frauenkirche',
  'munich_marienplatz',
  'munich_theatine_church',
  'salzburg_andrakirche',
  'salzburg_recthe_altstadt',
  'udine_devils_bridge',
  'udine_fagagna_church',
  'udine_villalta_castle',
  'vienna_state_opera'])

In [31]:
def build_k(camera):
    """ Build intrinsic matrix from pycolmap camera object """
    model = camera.model.name
    params = camera.params
    
    if model == 'SIMPLE_RADIAL':
        f, cx, cy, k = params
        K = np.array([[f, 0, cx],
                      [0, f, cy],
                      [0, 0, 1]])
    elif model == 'PINHOLE':
        fx, fy, cx, cy = params
        K = np.array([[fx, 0, cx],
                      [0, fy, cy],
                      [0, 0, 1]])
    elif model == 'SIMPLE_PINHOLE':
        f, cx, cy = params
        K = np.array([[f, 0, cx],
                      [0, f, cy],
                      [0, 0, 1]])
    else:
        raise NotImplementedError(f"Camera model {model} not implemented")
    
    return K

In [32]:
import pandas as pd

mixed_scenes = ["udine_devils_bridge", "udine_fagagna_church", "udine_villalta_castle"]

files = []
for scene_name in tqdm(scenes):
    rec_path = f"{data_path}/{scene_name}/colmap/sparse/0"
    if not os.path.exists(rec_path):
        print(f"Reconstruction path {rec_path} does not exist. Skipping scene {scene_name}.")
        continue
    rec = pycolmap.Reconstruction(rec_path)
    if True: #scene_name not in mixed_scenes:
        with open(f"{data_path}/{scene_name}/colmap/viewgraph_30.txt", "r") as f:
            lines = f.read().splitlines()
        pairs = [line.split() for line in lines]
    # else:
    #     df_pairs = pd.read_csv(f"{data_path}/{scene_name}/cyclic_depth_filtering_results1600_bidirectionally_filtered.csv")
    #     df_pairs = df_pairs[df_pairs['1px'] < 0.3] # filter out pairs with less than 100 matches, since we don't have the number of matches for the mixed scenes
    #     df_pairs['pair_key'] = df_pairs.apply(lambda row: tuple(sorted([row['level_0'], row['level_1']])), axis=1)
    #     df_unique = df_pairs.drop_duplicates(subset=['pair_key']).drop(columns=['pair_key'])
    #     # sort by level_0 and then by level_1 to ensure consistent ordering
    #     df_unique = df_unique.sort_values(by=['level_0', 'level_1']).reset_index(drop=True)
    #     # drop a row every other
    #     df_unique = df_unique.iloc[::2, :].reset_index(drop=True)
    #     pairs = df_unique[["level_0", "level_1"]].values.tolist()

    pairs = np.array(pairs)

    for pair in pairs:
        if True: #scene_name not in mixed_scenes:
            img1, img2, matches = pair[0].item(), pair[1].item(), int(pair[2].item())
        else:
            img1, img2 = pair[0].item(), pair[1].item()
            matches = 101 # dummy value to keep all pairs, since we don't have the number of matches for the mixed scenes

        if ('aerial' in img1 and not 'aerial' in img2) or (not 'aerial' in img1 and 'aerial' in img2):
            pass # keep all mixed pairs
        elif (matches < 100 or matches > 500):
            continue # skip too easy/hard pairs 
        try:
            K1 = build_k(rec.find_image_with_name(img1).camera)
            K2 = build_k(rec.find_image_with_name(img2).camera)

            R1 = rec.find_image_with_name(img1).cam_from_world.rotation.matrix()
            t1 = rec.find_image_with_name(img1).cam_from_world.translation
            R2 = rec.find_image_with_name(img2).cam_from_world.rotation.matrix()
            t2 = rec.find_image_with_name(img2).cam_from_world.translation

            R, t = compute_relative_camera_motion(R1, t1, R2, t2)
            R, t = R.numpy(), t.numpy()

            # Store line
            files.append(f"{scene_name}/frames/{img1} {scene_name}/frames/{img2} " +
                         " ".join(map(str, K1.flatten())) + " " +
                         " ".join(map(str, K2.flatten())) + " " +
                         " ".join(map(str, R.flatten())) + " " +
                         " ".join(map(str, t.flatten())))

    
        except Exception as e:
            print(f"Error processing pair ({img1}, {img2}): {e}")
            continue
    print(f"Scene {scene_name}: {len(pairs)} pairs, {len(files)} total pairs so far.")

print(f"Writing {len(files):,} pairs to {data_path}/pairs_calibrated.txt")

  0%|          | 0/12 [00:00<?, ?it/s]

Scene graz_church: 18294 pairs, 6184 total pairs so far.
Scene graz_townhall: 17644 pairs, 11073 total pairs so far.
Scene graz_university: 20917 pairs, 17725 total pairs so far.
Scene munich_frauenkirche: 3388 pairs, 18672 total pairs so far.
Scene munich_marienplatz: 9001 pairs, 22521 total pairs so far.
Scene munich_theatine_church: 3726 pairs, 24072 total pairs so far.
Scene salzburg_andrakirche: 4281 pairs, 26044 total pairs so far.
Scene salzburg_recthe_altstadt: 1378 pairs, 26508 total pairs so far.
Scene udine_devils_bridge: 13233 pairs, 33469 total pairs so far.
Scene udine_fagagna_church: 6367 pairs, 36447 total pairs so far.
Scene udine_villalta_castle: 14138 pairs, 42590 total pairs so far.
Scene vienna_state_opera: 3963 pairs, 43720 total pairs so far.
Writing 43,720 pairs to /home/mattia/Desktop/datasets/mydataset/data_test/pairs_calibrated.txt


In [33]:
dist = {"ground": 0, "aerial": 0, "mixed": 0, "tot":0}
for line in files:
    img1, img2 = line.split()[:2]
    if 'aerial' in img1 and 'aerial' in img2:
        dist['aerial'] += 1
    elif 'aerial' not in img1 and 'aerial' not in img2:
        dist['ground'] += 1
    else:
        dist['mixed'] += 1
    dist['tot'] += 1   
dist = {k: v/dist['tot']*100 for k, v in dist.items() if k != 'tot'}
print(f"Distribution of pairs: {dist}")

Distribution of pairs: {'ground': 74.83531564501372, 'aerial': 14.428179322964318, 'mixed': 10.736505032021958}


In [34]:
with open(f"{data_path}/pairs_calibrated.txt", "w") as f:
    f.write("# paths, intrisics, relative pose\n")
    f.write("# scene/image_1_path scene/image_2_path (fx1 0 cx1 0 fy1 cy1 0 0 1) (fx2 0 cx2 0 fy2 cy2 0 0 1) (R00 R01 R02 R10 R11 R12 R20 R21 R22) (tx ty tx)\n")
    f.write("\n".join(files))

In [35]:
# generate views.txt
files = []
images_set = ()
for scene_name in tqdm(scenes):
    rec_path = f"{data_path}/{scene_name}/colmap/sparse/0"
    if not os.path.exists(rec_path):
        print(f"Reconstruction path {rec_path} does not exist. Skipping scene {scene_name}.")
        continue
    rec = pycolmap.Reconstruction(rec_path)

    for image in rec.images.values():     
        try:   
            R = rec.find_image_with_name(image.name).cam_from_world.rotation.matrix()
            t = rec.find_image_with_name(image.name).cam_from_world.translation

            h = rec.find_image_with_name(image.name).camera.height
            w = rec.find_image_with_name(image.name).camera.width
            model = rec.find_image_with_name(image.name).camera.model.name
            params = rec.find_image_with_name(image.name).camera.params

            # Store line
            files.append(f"{scene_name}/frames/{image.name} " +
                " ".join(map(str, R.flatten())) + " " +
                " ".join(map(str, t.flatten())) + " " +
                f"{model} {w} {h} " +
                " ".join(map(str, params)))
    
    
        except Exception as e:
            print(f"Error processing pair ({img1}, {img2}): {e}")
            continue

print(f"Writing {len(files):,} views to {data_path}/views.txt")

  0%|          | 0/12 [00:00<?, ?it/s]

Writing 3,024 views to /home/mattia/Desktop/datasets/mydataset/data_test/views.txt


In [36]:
with open(f"{data_path}/views.txt", "w") as f:
    f.write("# scene/image_path (R00 R01 R02 R10 R11 R12 R20 R21 R22) (tx ty tz) camera_model width height camera_params\n")
    f.write("\n".join(files))

In [37]:
.

SyntaxError: invalid syntax (1933637684.py, line 1)

## covert megadepth view or ait2ground to md1500 format

In [ ]:
# views.txt format
# image_path R(3x3).flatten() t(3).flatten() model W H fx fy cx cy

# pairs_calibrated.txt format
# image1_path image2_path K1(3x3).flatten() K2(3x3).flatten() R_rel(3x3).flatten() t_rel(3).flatten()

In [ ]:
import numpy as np
from PIL import Image

path = Path("benchmarks/megadepth_air2ground/data")

index = np.load(path / "indices.npz", allow_pickle=True)

In [ ]:
index['pair_info']

In [ ]:
path = Path("benchmarks/megadepth_air2ground/data")

out = {}
for pair in index['pair_info']:
    img1_name = pair['pair_names'][0]
    if img1_name not in out:
        pose1 = pair['pose'][0]
        R1 = pose1[:3, :3]
        t1 = pose1[:3, 3]
        K1 = pair['intrinsic'][0]
        fx1, fy1, cx1, cy1 = K1[0, 0], K1[1, 1], K1[0, 2], K1[1, 2]
        model = "PINHOLE"
        w, h = Image.open(path / 'images' / img1_name).size

        s1 = f"{img1_name} {' '.join(map(str, R1.flatten()))} {' '.join(map(str, t1.flatten()))} {model} {w} {h} {fx1} {fy1} {cx1} {cy1}"
        out[img1_name] = s1

    img2_name = pair['pair_names'][1]
    if img2_name not in out:
        pose2 = pair['pose'][1]
        R2 = pose2[:3, :3]
        t2 = pose2[:3, 3]
        K2 = pair['intrinsic'][1]
        fx2, fy2, cx2, cy2 = K2[0, 0], K2[1, 1], K2[0, 2], K2[1, 2]
        model = "PINHOLE"
        img2_name = pair['pair_names'][1]
        w, h = Image.open(path / 'images' / img2_name).size

        s2 = f"{img2_name} {' '.join(map(str, R2.flatten()))} {' '.join(map(str, t2.flatten()))} {model} {w} {h} {fx2} {fy2} {cx2} {cy2}"
        out[img2_name] = s2

out = sorted(out.values())
with open(path / "views.txt", "w") as f:
    f.write("\n".join(out) + "\n")

In [ ]:
out = []
for pair in index['pair_info']:
    img1_name = pair['pair_names'][0]
    pose1 = pair['pose'][0]
    R1 = pose1[:3, :3]
    t1 = pose1[:3, 3]
    K1 = pair['intrinsic'][0]

    img2_name = pair['pair_names'][1]
    pose2 = pair['pose'][1]
    R2 = pose2[:3, :3]
    t2 = pose2[:3, 3]
    K2 = pair['intrinsic'][1]

    out.append(f"{img1_name} {img2_name} {' '.join(map(str, K1.flatten()))} {' '.join(map(str, K2.flatten()))} {' '.join(map(str, (R2 @ R1.T).flatten()))} {' '.join(map(str, (t2 - R2 @ R1.T @ t1).flatten()))}")

out = sorted(out)

In [ ]:
out = sorted(out)
with open(path / "pairs_calibrated.txt", "w") as f:
    f.write("\n".join(out) + "\n")